In [8]:
# Importing from tapas
import tapas.datasets
import tapas.generators
import tapas.attacks
import tapas.threat_models

In [9]:
# Try to use an executable as generator
from tapas.generators import GeneratorFromExecutable
import pandas as pd
import json

# Used to ensure attributes are correctly set.
class AttributeInterface:
    @property
    def name(self):
        raise NotImplementedError("Subclasses must implement this property")
    
    @property
    def type(self):
        raise NotImplementedError("Subclasses must implement this property")
    
    @property
    def representation(self):
        raise NotImplementedError("Subclasses must implement this property")
    
# Instance of the class so attributes are not publicly accesible.
class Attribute(AttributeInterface):
    def __init__(self, name, type, representation):
        self._name = name
        self._type = type
        self._representation = representation
        

    @property
    def name(self):
        return self._name

    @property
    def type(self):
        return self._type
    
    @property
    def representation(self):
        return self._representation
    
    def to_dict(self):
        return {
            "name": self._name,
            "type": self._type,
            "representation": self._representation
        }
    
attribute=Attribute('age','real/non-negative','number')
attribute.to_dict()

{'name': 'age', 'type': 'real/non-negative', 'representation': 'number'}

In [10]:
import pandas as pd

def classify_numeric_column(df, column_name):    
    # Get the column
    column = df[column_name]

    # Check if the column is numeric
    if not pd.api.types.is_numeric_dtype(column):
        raise TypeError(f"Column '{column_name}' is not numeric.")

    dtype = column.dtype
    if pd.api.types.is_float_dtype(dtype):
        num_type = 'float'
    elif pd.api.types.is_integer_dtype(dtype):
        num_type = 'integer'
    else:
        raise TypeError(f"Column '{column_name}' is not of type float or integer.")
    
    # Check if values are negative or positive
    min_val = column.min()
    max_val = column.max()

    if min_val < 0 and max_val > 0:
        sign = 'mixed'  # Both negative and positive values
    else:
        sign = 'positive'

    if num_type == 'float':
        if sign=='mixed':
            type = 'real'
        else:
            type = 'real/non-negative'
        representation = 'number'
    else:
        type= 'countable/ordered'
        representation = 'integer'
    
    return type, representation
    
  

def generate_description_file_from_csv(file_path: str, categorical_columns: str,  save_dir: str=None, special_columns_dict: dict = None):
    
    # Get file name
    file_name = file_path.split('/')[-1].split('.')[0]
    print(file_name)
    
    # Read file as Pandas DataFrame
    data = pd.read_csv(file_path, na_values='?')
    
    
    json_description = []
    
    special_columns=[]
    if special_columns_dict != None:
        special_columns = [col for col in special_columns_dict.keys()]
        
    numerical_columns=[ col for col in data.columns if (col not in categorical_columns) and (col not in special_columns)]
    
    for col in data.columns:
        if col in categorical_columns:
            list_elements=list(data[col].unique())
            list_elements = [str(element) for element in list_elements]
            json_description.append(Attribute(col,'finite',list_elements).to_dict())
        else:
            type_col,representation = classify_numeric_column(data,col)
            json_description.append(Attribute(col,type_col,representation).to_dict())
        
    
        
        
    print(json_description)
    json_str = json.dumps(json_description,indent=4)
    
    print(json_str)
    
    # Writing to sample.json
    with open(f"{save_dir}/{file_name}.json", "w") as outfile:
        outfile.write(json_str)
    
        
generate_description_file_from_csv('data/adult_complete.csv', ['workclass','education-num','marital-status','occupation','relationship','race','sex','native-country','income'],save_dir='data')    
    

data = tapas.datasets.TabularDataset.read(
     "data/adult_complete", label="adult"
 )

exe = GeneratorFromExecutable("../dist/generator_example_2.exe")
#exe.fit(data)
#ds=exe.generate(10)

adult_complete
[{'name': 'age', 'type': 'countable/ordered', 'representation': 'integer'}, {'name': 'workclass', 'type': 'finite', 'representation': ['State-gov', 'Self-emp-not-inc', 'Private', 'Federal-gov', 'Local-gov', 'nan', 'Self-emp-inc', 'Without-pay', 'Never-worked']}, {'name': 'education-num', 'type': 'finite', 'representation': ['13', '9', '7', '14', '5', '10', '12', '11', '4', '16', '15', '3', '6', '2', '1', '8']}, {'name': 'marital-status', 'type': 'finite', 'representation': ['Never-married', 'Married-civ-spouse', 'Divorced', 'Married-spouse-absent', 'Separated', 'Married-AF-spouse', 'Widowed']}, {'name': 'occupation', 'type': 'finite', 'representation': ['Adm-clerical', 'Exec-managerial', 'Handlers-cleaners', 'Prof-specialty', 'Other-service', 'Sales', 'Craft-repair', 'Transport-moving', 'Farming-fishing', 'Machine-op-inspct', 'Tech-support', 'nan', 'Protective-serv', 'Armed-Forces', 'Priv-house-serv']}, {'name': 'relationship', 'type': 'finite', 'representation': ['Not-i

In [11]:
data = tapas.datasets.TabularDataset.read("data/AML_simplified_states")

In [20]:
exe.fit(data)
ds=exe.generate(10)

In [21]:
ds.data

,simplified_class,treatment,STATUS_OS,Overall survival,Relative Overall survival,WHO 2016,WHO 2022,ICC 2022,ELN 2017,ELN 2022,...,SF3B1,SH2B3,SRSF2,STAG2,TET2,TP53,U2AF1,U2AF2,WT1,ZRSR2
0,2,Autologous HSCT,0,1442,1281,1,2,3,favorable,favorable,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,Allogeneic HSCT,1,514,260,9,10,12,favorable,favorable,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,Autologous HSCT,1,846,217,9,10,12,favorable,favorable,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,No treatment,0,1606,1606,3,1,1,favorable,na,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,No treatment,1,979,979,14,12,15,intermediate,adverse,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
5,1,Allogeneic HSCT,0,1338,882,10,11,13,favorable,favorable,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,1,Allogeneic HSCT,0,2834,2258,9,10,12,favorable,intermediate,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1,No treatment,0,3634,3634,3,1,1,favorable,na,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0,Autologous HSCT,0,2649,2506,14,14,16,adverse,intermediate,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,3,No treatment,1,12,12,3,1,1,favorable,na,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
data2=pd.read_csv('data/AML_simplified_states.csv')

In [24]:
data2[data2['Overall survival']==1442]

,simplified_class,treatment,STATUS_OS,Overall survival,Relative Overall survival,WHO 2016,WHO 2022,ICC 2022,ELN 2017,ELN 2022,...,SF3B1,SH2B3,SRSF2,STAG2,TET2,TP53,U2AF1,U2AF2,WT1,ZRSR2
60,2,Autologous HSCT,0,1442,1281,1,2,3,favorable,favorable,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
data_path = 'Data/AML_simplified_states.csv'
real_data = pd.read_csv(data_path)
numerical_columns=['AGE','Overall survival','Relative Overall survival','HB', 'PLT', 'WBC', 'LDH', 'BM BLASTS', 'PB BLASTS']
categorical_columns=[ col for col in real_data.columns if col not in numerical_columns]
generate_description_file_from_csv('data/AML_simplified_states.csv',categorical_columns=categorical_columns,save_dir='data')

AML_simplified_states
[{'name': 'simplified_class', 'type': 'finite', 'representation': ['3', '2', '0', '1']}, {'name': 'treatment', 'type': 'finite', 'representation': ['No treatment', 'Autologous HSCT', 'Allogeneic HSCT']}, {'name': 'STATUS_OS', 'type': 'finite', 'representation': ['1', '0']}, {'name': 'Overall survival', 'type': 'countable/ordered', 'representation': 'integer'}, {'name': 'Relative Overall survival', 'type': 'countable/ordered', 'representation': 'integer'}, {'name': 'WHO 2016', 'type': 'finite', 'representation': ['12', '13', '11', '9', '14', '1', '10', '2', '4', '6', '3', '5', '8']}, {'name': 'WHO 2022', 'type': 'finite', 'representation': ['12', '15', '14', '10', '2', '9', '7', '11', '3', '8', '1', '4', '6']}, {'name': 'ICC 2022', 'type': 'finite', 'representation': ['14', '16', '15', '12', '17', '3', '6', '13', '4', '5', '8', '9', '1', '7', '10', '11']}, {'name': 'ELN 2017', 'type': 'finite', 'representation': ['adverse', 'intermediate', 'favorable', 'na']}, {'na

In [13]:
data_path = 'Data/AML_simplified_states'
data = tapas.datasets.TabularDataset.read(
     data_path, label="AML"
 )

#exe = GeneratorFromExecutable("../dist/generator_example.exe")
#exe.fit(data)
#ds=exe.generate(10)

In [40]:
from subprocess import PIPE
import os
# Set an environment variable
os.environ["MLFLOW_URL"] = "http://127.0.0.1:5000"
os.environ["MODEL_URL"] = "models:/BN_trained_model/3"

In [41]:
import subprocess
proc = subprocess.Popen(["../dist/generator_example.exe", f"{1000}"], stdin = PIPE, stdout = PIPE, stderr=PIPE)

In [42]:
input = bytes(data.write_to_string(), 'utf-8')

In [43]:
output,errors = proc.communicate(input = input)

In [44]:
errors

b'2024/12/03 11:05:31 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model\'s dependencies and the current Python environment:\r\n - datasynthesizer (current: uninstalled, required: datasynthesizer==0.1.11)\r\n - scikit-learn (current: uninstalled, required: scikit-learn==1.5.2)\r\nTo fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model\'s environment and install dependencies using the resulting environment file.\r\nTraceback (most recent call last):\r\n  File "generator_example.py", line 76, in <module>\r\n    instance.generate(num_samples)\r\n  File "generator_example.py", line 53, in generate\r\n    loaded_model = mlflow.pyfunc.load_model(model_uri)\r\n                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\r\n  File "mlflow\\tracing\\provider.py", line 253, in wrapper\r\n  File "mlflow\\pyfunc\\__init__.py", line 1066, in load_model\r\n  File "mlflow\\pyfunc\\__init__.py", line 1051, in load_model\r\n  Fi

In [45]:
import subprocess
import os

# Set an environment variable
os.environ["MLFLOW_URL"] = "http://127.0.0.1:5000"
os.environ["MODEL_URL"] = "models:/BN_trained_model/3"

conda_env = 'mlflow_env'
result = subprocess.run(
    ["conda", "run", "--name",conda_env,"python", "generator_example.py", "1000"],# Pass your input data here
    text=True,
    stdout=subprocess.PIPE,  # Capture standard output
    stderr=subprocess.PIPE,  # Capture standard error
)

In [46]:
result.stderr

''

In [47]:
output=result.stdout

In [49]:
import mlflow

In [51]:
mlflow.set_registry_uri('http://127.0.0.1:5000')
loaded_model = mlflow.pyfunc.load_model("models:/BN_trained_model/3")

2024/12/03 11:22:19 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - datasynthesizer (current: uninstalled, required: datasynthesizer==0.1.11)
 - pandas (current: 1.5.3, required: pandas==2.2.3)
 - psutil (current: 6.0.0, required: psutil==5.9.4)
 - scikit-learn (current: 1.5.0, required: scikit-learn==1.5.2)
 - scipy (current: 1.13.1, required: scipy==1.14.1)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2024/12/03 11:22:19 WARNING mlflow.pyfunc: The version of Python that the model was saved in, `Python 3.11.9`, differs from the version of Python that is currently running, `Python 3.9.20`, and may be incompatible


TypeError: code() takes at most 16 arguments (18 given)

In [ ]:
from tapas.datasets.utils import get_dtype

get_dtype()

In [ ]:
# Options of Python to load the generators 

# Most of the methods need a generator to run the attacks. However, we are not sure how we will have access to the model.

# The options available is through an executable.

# Possible attacks will be implemented further right now we have the following in the library:

# Shadow modelling attacks
# Local neighborhood attacks 
# Inference data attacks


In [5]:
# Options to define the attacker Knowledge on Data 






In [6]:
# Option on the attacker knowledge on the Generator

# Black Box Knowledge

# No Box Knowledge

tapas.threat_models.NoBoxKnowledge(generator, num_synthetic_records=100, )


# Still required the generator and the number of synthetic records

NameError: name 'tapas' is not defined